# Bài 4.5 · Zero-shot với CLIP và GPT-2

**Câu hỏi của buổi này:** dùng một model **không học thêm gì** trên dữ liệu của bài thì làm được đến đâu, và phải đo nó thế nào cho công bằng?

| Mục | Sẽ thấy |
|---|---|
| 1. CLIP phân loại từ tên lớp | ảnh kèm ba lớp CLIP chọn; accuracy theo cách viết prompt |
| 2. Zero-shot hay few-shot | cần bao nhiêu ảnh có nhãn mỗi lớp để vượt zero-shot |
| 3. CLIP hay nhầm gì | cặp lớp hay nhầm, kèm ảnh |
| 4. GPT-2 chọn đoạn kết (HellaSwag) | một câu hỏi với bốn điểm số; tổng log và trung bình log |
| 5. Perplexity phụ thuộc giao thức | cùng model, cùng văn bản, năm con số khác nhau |
| 6. Tóm tắt | một bảng các con số chính |

Các dòng ✓ là phép kiểm in ra: chạy lại mà thấy ✗ thì có gì đó sai. Bật **GPU** và **Internet** trong *Session options*; chạy hết khoảng 8 phút. Bài giảng: [ai-canuc80k · bài 4.5](https://ai-canuc80k.ai-web-deploy-20261010.workers.dev/#/lesson/4.5).

In [ ]:
# Ô tiện ích: định dạng bảng, dòng kiểm ✓/✗, nhãn số trên biểu đồ. Không cần đọc để theo bài.
import warnings, contextlib, io, logging
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, HTML

plt.rcParams.update({
    'figure.dpi': 110, 'figure.figsize': (8, 4), 'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25,
    'legend.frameon': False, 'image.cmap': 'viridis',
})
C = {'blue': '#3b6ea8', 'clay': '#c2603e', 'sand': '#b8892b', 'green': '#4d7337', 'gray': '#8a877f'}
ORDER = [C['blue'], C['clay'], C['sand'], C['green'], C['gray']]
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=ORDER)


def table(df, title=None, fmt=None, best=None, lower_is_better=False, index=False):
    """In một bảng gọn: định dạng số, tô đậm ô tốt nhất của cột `best`."""
    st = df.style.format(fmt, na_rep='–')
    if best is not None:
        col = df[best]
        target = col.min() if lower_is_better else col.max()
        st = st.apply(lambda s: ['font-weight:700;background:#f5e6dd' if (s.name == best and v == target) else '' for v in s], axis=0)
    text_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if text_cols:
        st = st.set_properties(subset=text_cols, **{'text-align': 'left'})
    if not index:
        st = st.hide(axis='index')
    st = st.set_table_styles([
        {'selector': 'th', 'props': 'text-align:left;padding:6px 12px;border-bottom:2px solid #d3cebf;font-weight:600'},
        {'selector': 'td', 'props': 'padding:5px 12px;border-bottom:1px solid #e6e2d6'},
        {'selector': 'caption', 'props': 'caption-side:top;text-align:left;font-weight:700;font-size:14px;padding:4px 0 8px;white-space:nowrap'},
    ])
    if title:
        st = st.set_caption(title)
    display(st)


def check(msg, ok):
    """Dòng kiểm tra in ra: ✓ khi đúng, ✗ khi sai (không dừng notebook)."""
    print(('✓ ' if ok else '✗ ') + msg)


def bar_labels(ax, fmt='{:.2f}'):
    """Ghi giá trị lên đầu mỗi thanh của biểu đồ cột."""
    for cont in ax.containers:
        ax.bar_label(cont, labels=[fmt.format(v) for v in cont.datavalues], padding=3, fontsize=9)


@contextlib.contextmanager
def quiet():
    """Ẩn thanh tiến trình và dòng thông báo khi tải dữ liệu hay trọng số."""
    logging.disable(logging.WARNING)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)


def note(text):
    """Một dòng nhận xét in đậm nhẹ dưới output, tính từ số thật của lần chạy."""
    display(HTML(f'<div style="border-left:3px solid #c2603e;padding:4px 10px;margin:6px 0;color:#444">{text}</div>'))

In [ ]:
import time, math
import torch, transformers, datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, AutoModelForCausalLM
transformers.logging.set_verbosity_error(); datasets.disable_progress_bars()

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TEST, N_HELLA, N_TOKENS = 2000, 500, 30000
table(pd.DataFrame([('torch', torch.__version__), ('transformers', transformers.__version__),
                    ('thiết bị', torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU')],
                   columns=['thành phần', 'phiên bản']), 'Môi trường chạy')

## 1. CLIP: từ tên lớp đến bộ phân loại

CLIP có hai encoder, một cho ảnh, một cho chữ, học sao cho ảnh và chú thích đúng của nó có embedding gần nhau. Muốn phân loại: mã hóa mỗi **tên lớp** thành một vector chữ, mã hóa ảnh, rồi chọn lớp có cosine lớn nhất. Không có bước train nào.

In [ ]:
with quiet():
    clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
    proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
    cifar = datasets.load_dataset('uoft-cs/cifar10')

class HFImages:
    def __init__(self, split):
        self.d, self.targets, self.classes = split, list(split['label']), split.features['label'].names
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d[int(i)]
        return r['img'].convert('RGB'), r['label']
test, train = HFImages(cifar['test']), HFImages(cifar['train'])
classes = test.classes
VI = {'airplane': 'máy bay', 'automobile': 'ô tô', 'bird': 'chim', 'cat': 'mèo', 'deer': 'hươu',
      'dog': 'chó', 'frog': 'ếch', 'horse': 'ngựa', 'ship': 'tàu thủy', 'truck': 'xe tải'}
vi = [VI[c] for c in classes]
rng = np.random.default_rng(0)
te_idx = rng.permutation(len(test))[:N_TEST]
y_te = np.array(test.targets)[te_idx]
feat = lambda out: out if torch.is_tensor(out) else out.pooler_output          # transformers 5 trả object

def image_emb(ds, idx, batch=256):
    out = []
    with torch.inference_mode():
        for s in range(0, len(idx), batch):
            x = proc(images=[ds[i][0] for i in idx[s:s + batch]], return_tensors='pt')['pixel_values'].to(device)
            out.append(torch.nn.functional.normalize(feat(clip.get_image_features(pixel_values=x)), dim=1).cpu())
    return torch.cat(out)

def text_emb(prompts):
    with torch.inference_mode():
        t = proc(text=prompts, return_tensors='pt', padding=True).to(device)
        return torch.nn.functional.normalize(feat(clip.get_text_features(**t)), dim=1).cpu()

img_te = image_emb(test, te_idx)
templates = ['a photo of a {}.', 'a blurry photo of a {}.', 'a low resolution photo of a {}.',
             'a close-up photo of a {}.', 'a photo of the small {}.', 'a pixelated photo of a {}.']
ways = {'chỉ tên lớp ("cat")': text_emb(classes),
        'một câu ("a photo of a cat.")': text_emb([f'a photo of a {c}.' for c in classes]),
        'trung bình 6 câu': torch.nn.functional.normalize(torch.stack([text_emb([tp.format(c) for c in classes]) for tp in templates]).mean(0), dim=1)}
zs = pd.DataFrame([(n, ((img_te @ t.T).argmax(1).numpy() == y_te).mean()) for n, t in ways.items()], columns=['cách viết tên lớp', 'accuracy zero-shot'])
table(zs, f'CLIP zero-shot trên {N_TEST} ảnh test CIFAR-10', fmt={'accuracy zero-shot': '{:.3f}'}, best='accuracy zero-shot')
check('mọi cách đều vượt xa đoán bừa 0.1', (zs['accuracy zero-shot'] > 0.5).all())

In [ ]:
txt = ways['trung bình 6 câu']
prob = (100 * img_te @ txt.T).softmax(1)
pick = rng.choice(N_TEST, 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(17, 5.6), gridspec_kw={'height_ratios': [3, 2]})
for j, i in enumerate(pick):
    top = prob[i].topk(3)
    axes[0, j].imshow(test[te_idx[i]][0]); axes[0, j].axis('off'); axes[0, j].set_title(f'thật: {vi[y_te[i]]}', fontsize=10)
    cols = [C['green'] if int(k) == y_te[i] else C['gray'] for k in top.indices]
    axes[1, j].barh(range(3)[::-1], top.values, color=cols); bar_labels(axes[1, j])
    axes[1, j].set_yticks(range(3)[::-1], [vi[int(k)] for k in top.indices]); axes[1, j].set_xlim(0, 1.15); axes[1, j].grid(axis='y', visible=False)
fig.suptitle('Ba lớp CLIP chọn cho mỗi ảnh (xanh lá: lớp đúng), chỉ từ tên lớp', y=1.0)
plt.tight_layout(); plt.show()
note(f"Viết tên lớp thành câu giống chú thích ảnh và lấy trung bình nhiều câu: {zs.iloc[0, 1]:.3f} → <b>{zs.iloc[2, 1]:.3f}</b>, không học thêm gì.")

## 2. Zero-shot hay few-shot

Có vài ảnh có nhãn thì học được một linear probe trên chính embedding ảnh của CLIP. Cần bao nhiêu ảnh mỗi lớp để vượt zero-shot? Mỗi mức lặp 5 lần với các ảnh khác nhau.

In [ ]:
pool = rng.permutation(len(train))[:3000]
img_pool = image_emb(train, pool).numpy(); y_pool = np.array(train.targets)[pool]
few = []
for k in [1, 2, 4, 8, 16, 32]:
    accs = []
    for rep in range(5):
        r = np.random.default_rng(rep)
        sel = np.concatenate([r.choice(np.flatnonzero(y_pool == c), k, replace=False) for c in range(10)])
        accs.append((LogisticRegression(max_iter=3000).fit(img_pool[sel], y_pool[sel]).predict(img_te.numpy()) == y_te).mean())
    few.append((k, np.mean(accs), np.std(accs), np.min(accs)))
few = pd.DataFrame(few, columns=['ảnh mỗi lớp', 'acc trung bình', 'độ lệch chuẩn', 'lần tệ nhất'])
best_zs = zs['accuracy zero-shot'].max()
table(few, 'Linear probe trên embedding CLIP, 5 lần mỗi mức', fmt={'acc trung bình': '{:.3f}', 'độ lệch chuẩn': '{:.3f}', 'lần tệ nhất': '{:.3f}'})
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.errorbar(few['ảnh mỗi lớp'], few['acc trung bình'], few['độ lệch chuẩn'], marker='o', capsize=4, color=C['blue'], label='linear probe (± 1 độ lệch chuẩn)')
ax.axhline(best_zs, color=C['clay'], ls='--', label=f'zero-shot tốt nhất {best_zs:.3f}')
ax.set_xscale('log', base=2); ax.set_xticks(few['ảnh mỗi lớp'], few['ảnh mỗi lớp'])
ax.set(xlabel='số ảnh có nhãn mỗi lớp', ylabel='accuracy test', title='Bao nhiêu nhãn thì vượt zero-shot?'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()
beat = few[few['acc trung bình'] > best_zs]
note(f"1 ảnh mỗi lớp: {few.iloc[0, 1]:.3f}, lần tệ nhất chỉ {few.iloc[0, 3]:.3f}. "
     + (f"Từ <b>{int(beat.iloc[0, 0])}</b> ảnh mỗi lớp, linear probe mới vượt zero-shot." if len(beat) else f"Đến {few.iloc[-1, 0]} ảnh mỗi lớp, linear probe vẫn chưa vượt zero-shot."))

## 3. CLIP hay nhầm gì

In [ ]:
pred = prob.argmax(1).numpy()
cm = confusion_matrix(y_te, pred)
off = [(vi[a], vi[b], cm[a, b]) for a in range(10) for b in range(10) if a != b]
off = sorted(off, key=lambda t: -t[2])[:6]
table(pd.DataFrame(off, columns=['lớp thật', 'CLIP chọn', 'số ảnh']), 'Sáu cặp nhầm nhiều nhất')
a, b = vi.index(off[0][0]), vi.index(off[0][1])
ids = np.flatnonzero((y_te == a) & (pred == b))[:6]
fig, axes = plt.subplots(1, len(ids), figsize=(2.2 * len(ids), 2.6))
for ax, i in zip(np.atleast_1d(axes), ids):
    ax.imshow(test[te_idx[i]][0]); ax.axis('off'); ax.set_title(f'{vi[a]} → {vi[b]}', fontsize=10, color=C['clay'])
plt.tight_layout(); plt.show()
note('Ảnh 32 × 32 được phóng lên 224: các loài có dáng gần nhau ở độ phân giải thấp là chỗ CLIP nhầm nhiều nhất.')

## 4. GPT-2 chọn đoạn kết

Mỗi câu HellaSwag có một ngữ cảnh và bốn đoạn kết; chỉ một đoạn hợp lý. Model ngôn ngữ chấm mỗi đoạn kết bằng xác suất nó sinh ra đoạn ấy sau ngữ cảnh:

$$\log P(\text{kết} \mid \text{ngữ cảnh}) = \sum_t \log P(x_t \mid x_{<t}).$$

**Tổng log** thiên vị đoạn kết ngắn, vì mỗi token thêm vào cộng một số âm. **Trung bình log** (chia cho số token) bỏ được thiên vị ấy. Đoán bừa được 25%.

In [ ]:
with quiet():
    gtok = AutoTokenizer.from_pretrained('gpt2'); gpt = AutoModelForCausalLM.from_pretrained('gpt2').to(device).eval()
    hella = datasets.load_dataset('Rowan/hellaswag', split='validation').shuffle(seed=0).select(range(N_HELLA))

def ending_logprobs(ctx, ending):
    c = gtok(ctx, return_tensors='pt')['input_ids'][0]; e = gtok(' ' + ending, return_tensors='pt')['input_ids'][0]
    ids = torch.cat([c, e])[None].to(device)
    with torch.inference_mode():
        logp = gpt(ids).logits.log_softmax(-1)[0, :-1]
    lp = logp[torch.arange(ids.shape[1] - 1), ids[0, 1:]][len(c) - 1:]
    return lp.sum().item(), lp.mean().item(), len(e)

ex = hella[0]
sc = [ending_logprobs(ex['ctx'], e) for e in ex['endings']]
display(HTML(f"<div style='margin:4px 0'><b>Ngữ cảnh:</b> {ex['ctx']}</div>"))
table(pd.DataFrame([(('✓ ' if k == int(ex['label']) else '') + e, s[2], s[0], s[1]) for k, (e, s) in enumerate(zip(ex['endings'], sc))],
                   columns=['đoạn kết', 'số token', 'tổng log', 'trung bình log']), 'Một câu HellaSwag (✓: đoạn đúng)', fmt={'tổng log': '{:.1f}', 'trung bình log': '{:.2f}'})

t = time.time(); hit_sum = hit_mean = 0; L = []
for ex in hella:
    s = [ending_logprobs(ex['ctx'], e) for e in ex['endings']]; label = int(ex['label'])
    hit_sum += int(np.argmax([v[0] for v in s]) == label); hit_mean += int(np.argmax([v[1] for v in s]) == label)
    L.append((s[int(np.argmax([v[0] for v in s]))][2], s[int(np.argmax([v[1] for v in s]))][2], np.mean([v[2] for v in s])))
L = np.array(L)
hs = pd.DataFrame([('đoán bừa', 0.25, np.nan), ('tổng log', hit_sum / N_HELLA, L[:, 0].mean()), ('trung bình log', hit_mean / N_HELLA, L[:, 1].mean())],
                  columns=['cách chấm', 'accuracy', 'độ dài đoạn được chọn (token)'])
table(hs, f'GPT-2 (124 triệu tham số) trên {N_HELLA} câu HellaSwag ({time.time() - t:.0f} giây)', fmt={'accuracy': '{:.3f}', 'độ dài đoạn được chọn (token)': '{:.1f}'}, best='accuracy')
note(f"Độ dài trung bình của mọi đoạn kết là {L[:, 2].mean():.1f} token; tổng log chọn đoạn dài trung bình <b>{L[:, 0].mean():.1f}</b> token: đúng thiên vị đoạn ngắn. "
     f"GPT-2 nhỏ chỉ hơn đoán bừa một chút: zero-shot chỉ tốt khi model đủ lớn và bài gần với thứ nó đã học.")

## 5. Perplexity phụ thuộc giao thức

Perplexity là $\exp$ của trung bình $-\log P$ trên các token. Cùng model, cùng văn bản, nhưng nếu cắt văn bản thành các đoạn ngắn, mỗi token chỉ thấy ít ngữ cảnh phía trước nên đoán khó hơn. Hai con số perplexity chỉ so được khi cùng tokenizer, cùng văn bản, cùng cách cắt.

In [ ]:
with quiet():
    wiki = datasets.load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='test')
ids = gtok('\n\n'.join(list(wiki['text'])), return_tensors='pt')['input_ids'][0][:N_TOKENS]

def perplexity(window, stride=None):
    stride = stride or window
    nll, count, prev_end = 0.0, 0, 0
    for begin in range(0, len(ids) - 1, stride):
        end = min(begin + window, len(ids))
        x = ids[begin:end][None].to(device)
        with torch.inference_mode():
            logp = gpt(x).logits.log_softmax(-1)[0, :-1]
        lp = logp[torch.arange(x.shape[1] - 1), x[0, 1:]]
        new = end - max(prev_end, begin + 1)          # chỉ tính token chưa được chấm ở cửa sổ trước
        nll -= lp[-new:].sum().item(); count += new; prev_end = end
        if end == len(ids): break
    return math.exp(nll / count), count

rows = [(w, s or w, *perplexity(w, s)) for w, s in [(32, None), (128, None), (512, None), (1024, None), (1024, 512)]]
ppl = pd.DataFrame(rows, columns=['cửa sổ (token)', 'bước trượt', 'perplexity', 'số token được chấm'])
table(ppl, f'GPT-2 trên {N_TOKENS:,} token đầu của WikiText-2', fmt={'perplexity': '{:.2f}', 'số token được chấm': '{:,}'}, best='perplexity', lower_is_better=True)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(ppl['cửa sổ (token)'][:4], ppl['perplexity'][:4], marker='o', color=C['blue'], lw=2, label='cắt rời từng đoạn')
ax.scatter([1024], [ppl['perplexity'].iloc[4]], color=C['clay'], zorder=3, s=50, label='cửa sổ 1024 trượt 512')
for k, r in ppl.iterrows(): ax.annotate(f"{r['perplexity']:.1f}", (r['cửa sổ (token)'], r['perplexity']), xytext=(-34, -14) if k == 4 else (5, 5), textcoords='offset points', fontsize=9)
ax.set_xscale('log', base=2); ax.set(xlabel='độ dài cửa sổ (token)', ylabel='perplexity', title='Cùng model, cùng văn bản, khác cách cắt'); ax.legend()
plt.tight_layout(); plt.show()
check('cửa sổ trượt chấm đủ mọi token trừ token đầu', ppl['số token được chấm'].iloc[4] == N_TOKENS - 1)
note(f"Từ cửa sổ 32 tới cửa sổ 1024 trượt 512, perplexity đi từ <b>{ppl['perplexity'].iloc[0]:.1f}</b> xuống <b>{ppl['perplexity'].iloc[4]:.1f}</b> mà model không đổi.")

## 6. Tóm tắt

In [ ]:
summary = [('CLIP zero-shot: tên lớp / một câu / trung bình 6 câu', ' / '.join(f'{v:.3f}' for v in zs['accuracy zero-shot'])),
           ('linear probe với 1 / 32 ảnh mỗi lớp', f"{few.iloc[0, 1]:.3f} / {few.iloc[-1, 1]:.3f}"),
           ('cặp CLIP nhầm nhiều nhất', f'{off[0][0]} → {off[0][1]} ({off[0][2]} ảnh)'),
           ('HellaSwag: tổng log / trung bình log', f'{hit_sum / N_HELLA:.3f} / {hit_mean / N_HELLA:.3f}'),
           ('perplexity: cửa sổ 32 / 1024 trượt 512', f"{ppl['perplexity'].iloc[0]:.1f} / {ppl['perplexity'].iloc[4]:.1f}")]
table(pd.DataFrame(summary, columns=['điều đo', 'kết quả']), 'Các con số chính của notebook')

**Thử thêm khi dạy:** thêm vài câu tiếng Việt vào `templates` rồi so accuracy; đổi `gpt2` thành `gpt2-medium` rồi so HellaSwag; thêm tên lớp lạ ("a photo of a dinosaur") vào danh sách và xem ảnh nào bị kéo sang.